# 05 - PPO Training from Scratch (GPU Accelerated + Action Masking)

This notebook runs the full PPO training loop with:
1. **Vectorized multi-environment rollout collection** across **all 6 benchmark scenarios** (`mixed` domain randomization).
2. **Logit-level Action Masking** (prevents impossible placements with 0% probability).
3. **GPU Accelerated Tensor Updates** on CUDA (`NVIDIA GeForce RTX 3050 Laptop GPU`).
4. **Live training loss & cumulative reward plotting**.

In [5]:
%load_ext autoreload
%autoreload 2

import sys
import os
import torch
import numpy as np
import matplotlib.pyplot as plt

# Ensure project root is in python path
sys.path.insert(0, os.path.abspath('..'))

from rl.config import PPOConfig
from rl.trainer import PPOTrainer

print(f"PyTorch: {torch.__version__} | CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active Device: {torch.cuda.get_device_name(0)}")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
PyTorch: 2.7.1+cu118 | CUDA Available: True
Active Device: NVIDIA GeForce RTX 3050 Laptop GPU


## 1. Configure PPO Hyperparameters

In [6]:
config = PPOConfig(
    cluster_config="../configs/cluster_small.yaml",
    reward_config="../configs/reward.yaml",
    scenario="mixed",  # Parallel workers train across all 6 scenarios simultaneously
    num_envs=8,
    rollout_length=256,
    minibatch_size=128,
    epochs_per_update=10,
    total_timesteps=200000,  # 200,000 steps production run (~12 mins on RTX 3050)
    learning_rate=0.0003,
    device="cuda" if torch.cuda.is_available() else "cpu",
    checkpoint_dir="../checkpoints",
)

print("=== PPO Training Config ===")
for k, v in config.__dict__.items():
    print(f"  {k:22s}: {v}")

=== PPO Training Config ===
  cluster_config        : ../configs/cluster_small.yaml
  reward_config         : ../configs/reward.yaml
  scenario              : mixed
  max_queue_size        : 16
  sim_horizon_seconds   : 3600.0
  seed                  : 42
  learning_rate         : 0.0003
  gamma                 : 0.99
  gae_lambda            : 0.95
  clip_ratio            : 0.2
  entropy_coef          : 0.01
  value_coef            : 0.5
  max_grad_norm         : 0.5
  num_envs              : 8
  rollout_length        : 256
  minibatch_size        : 128
  epochs_per_update     : 10
  total_timesteps       : 200000
  hidden_dim            : 256
  device                : cuda
  checkpoint_dir        : ../checkpoints
  save_freq_steps       : 25000
  eval_freq_steps       : 10000
  eval_episodes         : 3


## 2. Execute Training Loop

In [ ]:
trainer = PPOTrainer(config)
history = trainer.train()

print("Training Complete!")

Starting PPO Training on [CUDA] with 8 Parallel Envs...
Total target timesteps: 200,000 (Batch size per update: 2048)

Step 010240/200000 | FPS: 205 | Mean Rew:  +29.78 | Loss(P): -0.0046 | Loss(V): 39.0198 | Entropy: 0.594 | KL: 0.0077
Step 020480/200000 | FPS: 200 | Mean Rew: +333.89 | Loss(P): -0.0096 | Loss(V): 110.9796 | Entropy: 0.544 | KL: 0.0085
Step 030720/200000 | FPS: 208 | Mean Rew: +141.53 | Loss(P): -0.0060 | Loss(V): 34.4957 | Entropy: 0.511 | KL: 0.0049
Step 040960/200000 | FPS: 211 | Mean Rew: +183.38 | Loss(P): -0.0053 | Loss(V): 37.4828 | Entropy: 0.521 | KL: 0.0044
Step 051200/200000 | FPS: 215 | Mean Rew: +193.11 | Loss(P): -0.0042 | Loss(V): 30.2254 | Entropy: 0.484 | KL: 0.0084
Step 061440/200000 | FPS: 210 | Mean Rew: +289.12 | Loss(P): -0.0102 | Loss(V): 67.7266 | Entropy: 0.508 | KL: 0.0056
Step 071680/200000 | FPS: 206 | Mean Rew: +123.00 | Loss(P): -0.0083 | Loss(V): 36.1552 | Entropy: 0.463 | KL: 0.0056
Step 081920/200000 | FPS: 206 | Mean Rew:  +79.84 | Lo

## 3. Training Telemetry & Loss Curves

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# 1. Mean Episode Reward
axes[0, 0].plot(history["timesteps"], history["mean_reward"], color='green', lw=2)
axes[0, 0].set_title("Mean Episode Reward")
axes[0, 0].set_xlabel("Timesteps")
axes[0, 0].grid(True, alpha=0.3)

# 2. Policy Loss (Clipped Surrogate)
axes[0, 1].plot(history["timesteps"], history["policy_loss"], color='crimson', lw=2)
axes[0, 1].set_title("Policy Loss (Clipped Surrogate)")
axes[0, 1].set_xlabel("Timesteps")
axes[0, 1].grid(True, alpha=0.3)

# 3. Value Function Loss (MSE)
axes[1, 0].plot(history["timesteps"], history["value_loss"], color='blue', lw=2)
axes[1, 0].set_title("Value Loss (Critic MSE)")
axes[1, 0].set_xlabel("Timesteps")
axes[1, 0].grid(True, alpha=0.3)

# 4. Policy Entropy
axes[1, 1].plot(history["timesteps"], history["entropy"], color='purple', lw=2)
axes[1, 1].set_title("Policy Entropy (Exploration)")
axes[1, 1].set_xlabel("Timesteps")
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()